# LMCA-TIC 录屏版执行 Notebook

本 Notebook 只对应当前录屏主线：

- 平台：ModelScope Notebook
- GPU：A10 24GB 单卡
- 模型：`Qwen2.5-0.5B-Instruct` 本地离线目录
- 数据：`ICEWS14` 缩小录屏版 `300/30/30`
- 目标：从数据切分、BIE、预处理、训练、评测到绘图的完整闭环

注意事项：

- 不要同时启动多个训练进程。
- 当前 `train` 命令内部会再次执行一次 `preprocess`，这属于现有实现行为，不是故障。
- 本 Notebook 默认工作目录为 `/mnt/workspace/LMCA-TIC`。


In [ ]:
import json
import os
import shlex
import subprocess
from pathlib import Path

WORKSPACE = Path("/mnt/workspace/LMCA-TIC")
FULL_RAW_DIR = WORKSPACE / "data/local/icews14/raw"
RECORD_RAW_DIR = WORKSPACE / "data/local/icews14_record_small/raw"
RECORD_BIE_DIR = WORKSPACE / "data/local/icews14_record_small/bie"
PROCESSED_DIR = WORKSPACE / "data/processed/icews14_record_small"
OUTPUT_DIR = WORKSPACE / "outputs/icews14_record_qwen25_05b_a10"
CHECKPOINT_DIR = WORKSPACE / "checkpoints/icews14_record_qwen25_05b_a10"
LOG_DIR = WORKSPACE / "logs/icews14_record_qwen25_05b_a10"
MODEL_DIR = WORKSPACE / "models/Qwen2.5-0.5B-Instruct"
CONFIG_PATH = WORKSPACE / "configs/experiments/icews14_record_qwen25_05b_a10.yaml"

os.chdir(WORKSPACE)
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

def rel(path: Path) -> str:
    return shlex.quote(str(path.relative_to(WORKSPACE)))

def run(cmd: str) -> None:
    print(f"$ {cmd}")
    subprocess.run(["bash", "-lc", cmd], check=True)

print("cwd =", WORKSPACE)
print("config =", CONFIG_PATH)
print("offline =", {k: os.environ[k] for k in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE")})


In [ ]:
required_paths = [
    FULL_RAW_DIR / "train.txt",
    FULL_RAW_DIR / "valid.txt",
    FULL_RAW_DIR / "test.txt",
    MODEL_DIR,
    CONFIG_PATH,
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required paths:\n" + "\n".join(missing))
print("Sanity check passed.")


## 1. 切分录屏版 `300/30/30`

这里使用确定性的前缀切分，目的是保证录屏流程稳定且可复现。


In [ ]:
run(f"mkdir -p {shlex.quote(str(RECORD_RAW_DIR))} {shlex.quote(str(RECORD_BIE_DIR))}")
run(f"sed -n '1,300p' {shlex.quote(str(FULL_RAW_DIR / 'train.txt'))} > {shlex.quote(str(RECORD_RAW_DIR / 'train.txt'))}")
run(f"sed -n '1,30p' {shlex.quote(str(FULL_RAW_DIR / 'valid.txt'))} > {shlex.quote(str(RECORD_RAW_DIR / 'valid.txt'))}")
run(f"sed -n '1,30p' {shlex.quote(str(FULL_RAW_DIR / 'test.txt'))} > {shlex.quote(str(RECORD_RAW_DIR / 'test.txt'))}")
run(
    "wc -l "
    + shlex.quote(str(RECORD_RAW_DIR / 'train.txt'))
    + " "
    + shlex.quote(str(RECORD_RAW_DIR / 'valid.txt'))
    + " "
    + shlex.quote(str(RECORD_RAW_DIR / 'test.txt'))
)


## 2. 构建离线 BIE

不依赖 Wikidata/Wikipedia，仅使用数据集内部统计和规则归纳。


In [ ]:
run(f"PYTHONPATH=src python3 -m lmca_tic.cli build-bie --config {rel(CONFIG_PATH)}")


## 3. 预处理

这一步会生成 `processed` 目录下的样本、实体、关系和 filtered target 工件。


In [ ]:
run(f"PYTHONPATH=src python3 -m lmca_tic.cli preprocess --config {rel(CONFIG_PATH)}")


## 4. 训练

训练过程中当前窗口会直接显示进度条、loss、lr、opt_steps、eta 和 GPU 诊断。


In [ ]:
run(f"PYTHONPATH=src python3 -m lmca_tic.cli train --config {rel(CONFIG_PATH)}")


## 5. 评测

训练结束后再单独显式跑一次 `evaluate`，便于录屏时稳定展示测试指标输出。


In [ ]:
run(
    f"PYTHONPATH=src python3 -m lmca_tic.cli evaluate --config {rel(CONFIG_PATH)} --checkpoint best.pt --split test"
)


## 6. 绘图

从 `train_history.jsonl` 生成 `training_curves.png`。


In [ ]:
run(
    f"PYTHONPATH=src python3 scripts/plot_training_curves.py --history {rel(OUTPUT_DIR / 'train_history.jsonl')} --output {rel(OUTPUT_DIR / 'training_curves.png')}"
)


## 7. 展示产物

这里读取测试指标，并尽量直接预览训练曲线，方便录屏收尾展示。


In [ ]:
run(f"ls -lah {shlex.quote(str(OUTPUT_DIR))}")

metrics = json.loads((OUTPUT_DIR / 'test_metrics.json').read_text(encoding='utf-8'))
print(json.dumps(metrics, indent=2, ensure_ascii=False))

important_artifacts = [
    OUTPUT_DIR / 'train_history.jsonl',
    OUTPUT_DIR / 'training_curves.png',
    OUTPUT_DIR / 'test_metrics.json',
    OUTPUT_DIR / 'graph_summary.json',
]
for artifact in important_artifacts:
    print(f"exists={artifact.exists()} path={artifact}")

try:
    from IPython.display import Image, display
    display(Image(filename=str(OUTPUT_DIR / 'training_curves.png')))
except Exception as exc:
    print(f"Image preview skipped: {exc}")
